![DataProjectLab](../../logo_dataprojectlab.png)


# Notebook 1 — Contexte, Installation SQL Server & Decouverte des Donnees
**SOLUTION COMPLETE**

**Projet :** HotelChain West Analytics  
**Periode :** Janvier 2022 — Juin 2024  
**Auteur :** DataProjectLab


---
## Contexte metier

HotelChain West est une chaine hoteliere premium operant dans 4 pays d'Afrique de l'Ouest avec **5 etablissements** (2 a Abidjan, 1 a Dakar, 1 a Douala, 1 a Accra) et **515 chambres**.

> *"Nos taux d'occupation varient fortement selon les saisons et les etablissements mais nous n'avons pas de visibilite claire sur les causes. Certains hotels sur-performent tandis que d'autres stagnent. Nous ne savons pas non plus quels canaux de reservation sont les plus rentables ni quels clients genèrent le plus de valeur."*  
> — Directeur General, HotelChain West

Ce notebook couvre trois parties :
1. Installation de l'environnement SQL Server
2. Creation de la base de donnees et chargement des CSV
3. Exploration et diagnostic qualite des donnees

---
## PARTIE 1 — Installation de l'environnement SQL Server

### Pourquoi SQL Server pour ce projet ?

SQL Server est le SGBD (Systeme de Gestion de Base de Donnees) de Microsoft, largement utilise en entreprise. Il offre :
- Des **window functions** avancees (RANK, LAG, LEAD, NTILE)
- Des **CTEs** (Common Table Expressions) pour structurer les requetes complexes
- Des **procedures stockees** pour encapsuler la logique metier
- Une **integration native avec Power BI** (connexion directe sans export)

La version **Developer Edition** est gratuite et identique a la version Enterprise — c'est le choix ideal pour apprendre.

### Option A — SQL Server + SSMS (recommande sur Windows)

**Etape 1 : Telecharger SQL Server Developer Edition**
```
https://www.microsoft.com/fr-fr/sql-server/sql-server-downloads
```
Choisir **Developer** → Cliquer **Telecharger maintenant** → Executer l'installeur → Choisir **Installation de base** → Accepter les termes → Choisir le dossier → **Installer**. Duree : environ 10-15 minutes.

**Etape 2 : Installer SSMS (SQL Server Management Studio)**
```
https://aka.ms/ssmsfullsetup
```
SSMS est l'interface graphique officielle. Elle permet d'ecrire des requetes, visualiser les tables, importer des donnees et gerer la base. Telecharger et executer l'installeur — options par defaut suffisent.

**Etape 3 : Ouvrir SSMS et se connecter**
```
Demarrer → SQL Server Management Studio
Fenetre de connexion :
  Type de serveur    : Moteur de base de donnees
  Nom du serveur     : localhost  (ou votre nom de machine)
  Authentification   : Authentification Windows
  Cliquer            : Connecter
```

Si la connexion echoue : verifier que le service **SQL Server (MSSQLSERVER)** est bien demarré dans les Services Windows (Win+R → services.msc).

### Option B — Extension VS Code (mssql)

Si vous preferez VS Code a SSMS :

**Etape 1 : Installer l'extension**
```
VS Code → Ctrl+Shift+X → Rechercher 'SQL Server (mssql)'
Editeur : Microsoft → Installer
```

**Etape 2 : Creer un profil de connexion**
```
Ctrl+Shift+P → taper 'MS SQL: Connect'
→ Create Connection Profile
→ Server name     : localhost
→ Database name   : HotelChainDB (apres creation)
→ Authentication  : Integrated Windows Authentication
→ Profile name    : HotelChain Local
```

**Etape 3 : Executer une requete**
```
Creer un fichier query.sql
Ecrire votre requete SQL
Ctrl+Shift+E pour executer
Les resultats s'affichent dans un panneau en bas
```

**Avantage VS Code :** coloration syntaxique avancee, IntelliSense SQL, integration Git native. Ideal si vous deja familier avec VS Code.

### Option C — Azure Data Studio (cross-platform)

Pour les utilisateurs Mac ou Linux :
```
https://aka.ms/azuredatastudio
```
Interface moderne et epuree, disponible sur Windows, macOS et Linux. Meme experience de connexion que SSMS mais avec un design plus moderne. Supporte les notebooks SQL directement dans l'interface.

---
## PARTIE 2 — Creation de la base et chargement des donnees

### Etape 1 — Creer la base HotelChainDB

### METHODE
En SQL Server, chaque projet doit avoir sa propre base de donnees. C'est l'equivalent d'un dossier qui contiendra toutes les tables du projet. La commande `USE HotelChainDB` dit a SQL Server : "toutes les requetes suivantes s'appliquent a cette base".

`GO` est un separateur de lots specifique a SQL Server (et SSMS). Il signifie : executer tout ce qui precede avant de continuer. Ce n'est pas du SQL pur — c'est une instruction propre a SSMS/sqlcmd.

In [ ]:
-- Executer dans SSMS (F5 ou bouton Execute)
CREATE DATABASE HotelChainDB;
GO

USE HotelChainDB;
GO

-- Verification
SELECT name, create_date
FROM sys.databases
WHERE name = 'HotelChainDB';


### Etape 2 — Creer les 6 tables

### METHODE
On cree les tables dans un ordre precis : les tables **referencees** (parents) avant les tables **referençantes** (enfants). En SQL, une cle etrangere (`REFERENCES`) ne peut pointer que vers une table qui existe deja.

**Ordre obligatoire ici :**
1. `hotels` — referencee par chambres et reservations
2. `chambres` — referencee par reservations
3. `clients` — referencee par reservations
4. `reservations` — referencee par paiements et services
5. `paiements` et `services` — en dernier

Les types de donnees choisis sont importants :
- `VARCHAR(n)` : texte de longueur variable (max n caracteres)
- `INT` : entier
- `DECIMAL(12,2)` : nombre decimal avec 2 chiffres apres la virgule (pour les montants)
- `DATE` : date sans heure
- `BIT` : booleen (0 ou 1)

In [ ]:
USE HotelChainDB;
GO

-- TABLE 1 : hotels (table parent)
CREATE TABLE hotels (
    hotel_id       VARCHAR(10)  PRIMARY KEY,
    nom            VARCHAR(100) NOT NULL,
    ville          VARCHAR(50)  NOT NULL,
    pays           VARCHAR(50)  NOT NULL,
    etoiles        INT          CHECK (etoiles BETWEEN 1 AND 5),
    nb_chambres    INT          NOT NULL,
    date_ouverture DATE,
    directeur      VARCHAR(100)
);

-- TABLE 2 : chambres (enfant de hotels)
CREATE TABLE chambres (
    chambre_id    VARCHAR(10)  PRIMARY KEY,
    hotel_id      VARCHAR(10)  NOT NULL REFERENCES hotels(hotel_id),
    numero        VARCHAR(10),
    type_chambre  VARCHAR(20)  CHECK (type_chambre IN ('Standard','Superieure','Deluxe','Suite')),
    capacite      INT,
    prix_nuit     DECIMAL(12,2) NOT NULL,
    etage         INT,
    vue_mer       BIT          DEFAULT 0,
    statut        VARCHAR(20)  DEFAULT 'Disponible'
);

-- TABLE 3 : clients (independante)
CREATE TABLE clients (
    client_id      VARCHAR(10)  PRIMARY KEY,
    prenom         VARCHAR(50),
    nom            VARCHAR(50),
    email          VARCHAR(100),
    nationalite    VARCHAR(50),
    genre          CHAR(1)      CHECK (genre IN ('M','F')),
    age            INT,
    telephone      VARCHAR(30),
    client_fidele  BIT          DEFAULT 0,
    nb_sejours_total INT        DEFAULT 0
);

-- TABLE 4 : reservations (enfant de hotels, chambres, clients)
CREATE TABLE reservations (
    reservation_id    VARCHAR(10)   PRIMARY KEY,
    hotel_id          VARCHAR(10)   REFERENCES hotels(hotel_id),
    chambre_id        VARCHAR(10)   REFERENCES chambres(chambre_id),
    client_id         VARCHAR(10)   REFERENCES clients(client_id),
    date_arrivee      DATE          NOT NULL,
    date_depart       DATE          NOT NULL,
    nb_nuits          INT,
    nb_adultes        INT,
    canal             VARCHAR(30),
    motif_sejour      VARCHAR(30),
    prix_nuit_remise  DECIMAL(12,2),
    montant_total     DECIMAL(12,2),
    statut            VARCHAR(20),
    csat              VARCHAR(5),    -- VARCHAR pour accepter les NULL
    date_reservation  DATE
);

-- TABLE 5 : paiements
CREATE TABLE paiements (
    paiement_id      VARCHAR(10)  PRIMARY KEY,
    reservation_id   VARCHAR(10)  REFERENCES reservations(reservation_id),
    date_paiement    DATE,
    montant          DECIMAL(12,2),
    moyen_paiement   VARCHAR(30),
    statut_paiement  VARCHAR(20)
);

-- TABLE 6 : services (extras)
CREATE TABLE services (
    service_id      VARCHAR(10)  PRIMARY KEY,
    reservation_id  VARCHAR(10)  REFERENCES reservations(reservation_id),
    hotel_id        VARCHAR(10)  REFERENCES hotels(hotel_id),
    date_service    DATE,
    categorie       VARCHAR(30),
    montant         DECIMAL(12,2),
    quantite        INT
);

PRINT 'Tables creees avec succes';


### Etape 3 — Importer les CSV avec BULK INSERT

### METHODE
`BULK INSERT` est la commande SQL Server la plus rapide pour charger des fichiers CSV en masse. Elle est bien plus performante qu'un `INSERT` ligne par ligne.

**Parametres cles :**
- `FIRSTROW = 2` : ignorer la premiere ligne (en-tetes de colonnes)
- `FIELDTERMINATOR = ','` : les colonnes sont separees par des virgules
- `ROWTERMINATOR = '\n'` : chaque ligne se termine par un saut de ligne
- `TABLOCK` : verrou au niveau de la table pour accelerer l'import

> **Important :** Adapter le chemin `C:\chemin\...` vers votre dossier `dataset\` reel. Sur Windows, utiliser les doubles backslashes `\\` ou les slashes `/`.

In [ ]:
-- Adapter le chemin vers votre dossier dataset/
-- Exemple : 'C:\Users\VotreNom\hotelchain\dataset\hotels.csv'

BULK INSERT hotels
FROM 'C:\chemin\vers\hotelchain\dataset\hotels.csv'
WITH (FIRSTROW=2, FIELDTERMINATOR=',', ROWTERMINATOR='0x0a', TABLOCK);

BULK INSERT chambres
FROM 'C:\chemin\vers\hotelchain\dataset\chambres.csv'
WITH (FIRSTROW=2, FIELDTERMINATOR=',', ROWTERMINATOR='0x0a', TABLOCK);

BULK INSERT clients
FROM 'C:\chemin\vers\hotelchain\dataset\clients.csv'
WITH (FIRSTROW=2, FIELDTERMINATOR=',', ROWTERMINATOR='0x0a', TABLOCK);

BULK INSERT reservations
FROM 'C:\chemin\vers\hotelchain\dataset\reservations.csv'
WITH (FIRSTROW=2, FIELDTERMINATOR=',', ROWTERMINATOR='0x0a', TABLOCK);

BULK INSERT paiements
FROM 'C:\chemin\vers\hotelchain\dataset\paiements.csv'
WITH (FIRSTROW=2, FIELDTERMINATOR=',', ROWTERMINATOR='0x0a', TABLOCK);

BULK INSERT services
FROM 'C:\chemin\vers\hotelchain\dataset\services.csv'
WITH (FIRSTROW=2, FIELDTERMINATOR=',', ROWTERMINATOR='0x0a', TABLOCK);

PRINT 'Import termine';


### Etape 4 — Verification du chargement

### METHODE
`UNION ALL` permet de combiner les resultats de plusieurs requetes en une seule sortie. Ici on l'utilise pour verifier le nombre de lignes de toutes les tables en une seule execution. `UNION ALL` (avec ALL) est plus rapide que `UNION` car il ne supprime pas les doublons.

In [ ]:
-- Verification du nombre de lignes par table
SELECT 'hotels'       AS table_name, COUNT(*) AS nb_lignes FROM hotels
UNION ALL
SELECT 'chambres',    COUNT(*) FROM chambres
UNION ALL
SELECT 'clients',     COUNT(*) FROM clients
UNION ALL
SELECT 'reservations',COUNT(*) FROM reservations
UNION ALL
SELECT 'paiements',   COUNT(*) FROM paiements
UNION ALL
SELECT 'services',    COUNT(*) FROM services
ORDER BY table_name;


### INTERPRETATION

Le chargement doit donner exactement :

| Table | Lignes attendues |
|---|---|
| hotels | **5** |
| chambres | **515** |
| clients | **3 000** |
| reservations | **8 000** |
| paiements | **9 698** |
| services | **9 001** |

Si un nombre est incorrect, verifier le chemin du fichier CSV et les parametres du BULK INSERT (separateur, encodage). Le total de 9 698 paiements > 8 000 reservations s'explique par les paiements en plusieurs tranches.

---
## PARTIE 3 — Exploration des donnees

### Etape 5 — Structure et relations

### METHODE
`INFORMATION_SCHEMA.COLUMNS` est une **vue systeme** de SQL Server qui liste toutes les colonnes de toutes les tables. C'est l'equivalent de `.info()` en pandas. Elle est standard SQL et fonctionne sur la plupart des SGBD (MySQL, PostgreSQL...).

`sp_help 'nom_table'` est une procedure systeme SQL Server plus detaillee : elle affiche les types de donnees, les contraintes, les cles etrangeres et les index.

In [ ]:
-- Structure complete de toutes les tables
SELECT
    TABLE_NAME     AS table_name,
    COLUMN_NAME    AS colonne,
    DATA_TYPE      AS type_donnee,
    CHARACTER_MAXIMUM_LENGTH AS longueur_max,
    IS_NULLABLE    AS nullable
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = 'dbo'
ORDER BY TABLE_NAME, ORDINAL_POSITION;

-- Detail d'une table specifique
EXEC sp_help 'reservations';


### INTERPRETATION — Schema de relations

Les cles de jointure identifiees :

```
hotels      ←→  chambres       hotel_id    (1 hotel → N chambres)
hotels      ←→  reservations   hotel_id    (1 hotel → N reservations)
chambres    ←→  reservations   chambre_id  (1 chambre → N reservations dans le temps)
clients     ←→  reservations   client_id   (1 client → N reservations)
reservations ←→  paiements    reservation_id (1 reservation → 1 ou 2 paiements)
reservations ←→  services     reservation_id (1 reservation → N services extras)
```

**Point important :** Une chambre apparait dans N reservations differentes (elle est reservee par differents clients successivement). La cle de jointure `chambre_id` ne garantit pas l'unicite dans `reservations` — c'est normal et attendu.

---
## Etape 6 — Diagnostic qualite

### METHODE
Le diagnostic qualite en SQL suit toujours la meme sequence :
1. **Nulls** : `SUM(CASE WHEN col IS NULL THEN 1 ELSE 0 END)`
2. **Doublons** : `GROUP BY col HAVING COUNT(*) > 1`
3. **Aberrants** : `WHERE valeur NOT BETWEEN borne_min AND borne_max`
4. **Incoherences** : conditions metier (montant negatif, nuits = 0)

`CASE WHEN ... THEN 1 ELSE 0 END` est le pattern fondamental pour compter des lignes conditionnellement en SQL. Retenir ce pattern — il reviendra constamment dans les requetes analytiques.

In [ ]:
-- 1. Valeurs NULL dans clients
SELECT
    SUM(CASE WHEN email        IS NULL THEN 1 ELSE 0 END) AS null_email,
    SUM(CASE WHEN age          IS NULL THEN 1 ELSE 0 END) AS null_age,
    SUM(CASE WHEN nationalite  IS NULL THEN 1 ELSE 0 END) AS null_nationalite,
    SUM(CASE WHEN telephone    IS NULL THEN 1 ELSE 0 END) AS null_telephone
FROM clients;


In [ ]:
-- 2. Doublons email dans clients
SELECT email, COUNT(*) AS nb_occurrences
FROM clients
WHERE email IS NOT NULL
GROUP BY email
HAVING COUNT(*) > 1
ORDER BY nb_occurrences DESC;


### INTERPRETATION — Doublons email

**3 emails dupliques detectes.** Chaque email apparait 2 fois dans la base clients. Ces doublons sont probablement des inscriptions multiples du meme client (depuis deux appareils differents, ou deux reservations distinctes). Si on ne les traite pas, les analyses 'par client' surcompteront ces personnes et fausseront les calculs de valeur client (CLV).

**Correction dans le Notebook 2 :** on creera une vue `vw_clients_propres` qui ne conserve que la premiere occurrence de chaque email.

In [ ]:
-- 3. Ages aberrants dans clients
SELECT client_id, prenom, nom, age
FROM clients
WHERE age < 16 OR age > 80 OR age IS NULL
ORDER BY age;


### INTERPRETATION — Ages aberrants

**4 ages aberrants detectes :**

| client_id | Age | Probleme |
|---|---|---|
| CLI0016 | -3 | Age negatif — impossible |
| CLI0801 | 0 | Age nul — non renseigne ou erreur |
| CLI0301 | 150 | Age impossible — erreur de saisie |
| CLI1201 | 200 | Age impossible — erreur de saisie |

Ces erreurs viennent probablement de formulaires sans validation cote client (l'utilisateur peut entrer n'importe quelle valeur). Impact : si on calcule l'age moyen sans correction, le resultat sera fausse. La correction standard est de remplacer par la **mediane** des ages valides (plus robuste que la moyenne face aux extremes).

In [ ]:
-- 4. Montants negatifs dans reservations
SELECT reservation_id, montant_total, statut, canal
FROM reservations
WHERE montant_total <= 0
ORDER BY montant_total;

-- 5. Reservations avec nb_nuits = 0
SELECT reservation_id, date_arrivee, date_depart, nb_nuits
FROM reservations
WHERE nb_nuits = 0;

-- 6. Montants negatifs dans paiements
SELECT paiement_id, reservation_id, montant, statut_paiement
FROM paiements
WHERE montant <= 0;


### INTERPRETATION — Incoherences reservations et paiements

**Reservations :**
- **5 montants negatifs** dans `reservations` (entre -10 000 et -50 000 FCFA). Ces valeurs sont techniquement impossibles : une reservation ne peut pas avoir un prix negatif. Il s'agit probablement d'erreurs d'import ou de remboursements mal enregistres dans le systeme source.
- **3 nb_nuits = 0** : des reservations sans nuit sont incoherentes avec le concept d'une reservation hoteliere. Ce sont probablement des reservations annulees le jour meme ou des erreurs de saisie.

**Paiements :**
- **10 montants negatifs ou nuls** detectes. Les valeurs negatives peuvent correspondre a des remboursements partiels saisis en negatif par erreur au lieu d'utiliser le statut 'Rembourse'.

**Regle metier appliquee dans le NB2 :**
Pour les analyses de revenu, on ne gardera que `montant > 0` et `statut = 'Valide'`. Les anomalies sont documentees mais pas supprimees des tables brutes — principe de conservation des donnees originales.

In [ ]:
-- 7. Distribution des statuts de reservation
SELECT
    statut,
    COUNT(*)                              AS nb_reservations,
    ROUND(COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER(), 1)        AS pourcentage
FROM reservations
GROUP BY statut
ORDER BY nb_reservations DESC;


### INTERPRETATION — Distribution des statuts

| Statut | Nombre | % |
|---|---|---|
| Terminee | **7 214** | **90.2%** |
| Confirmee | 315 | 3.9% |
| Annulee | 306 | **3.8%** |
| En cours | 165 | 2.1% |

**Lecture :** 90.2% des reservations sont terminees — c'est la base principale des analyses. Le taux d'annulation de **3.8%** est bon pour l'industrie (la norme secteur est entre 5% et 15%). Les 315 'Confirmees' et 165 'En cours' sont des reservations actives (periode juin 2024).

**Attention DAX :** dans Power BI, toujours filtrer `statut IN ('Terminee', 'En cours')` pour les analyses de revenu, et `statut = 'Terminee'` pour le CSAT (on ne peut noter qu'un sejour termine).

**Note SQL :** `SUM(COUNT(*)) OVER()` est une window function qui calcule le total sur toutes les lignes du GROUP BY. Diviser par ce total donne le pourcentage de chaque statut — pattern fondamental a retenir.

---
## Etape 7 — Premiers KPIs

### METHODE
Les KPIs (Key Performance Indicators) sont les indicateurs de pilotage fondamentaux du metier hotelier. Ils se calculent toujours sur les **donnees propres** — ici les reservations terminees et les paiements valides.

On utilise une seule requete avec plusieurs `SUM(CASE WHEN...)` pour eviter les jointures multiples et garantir la coherence des calculs.

In [ ]:
-- KPI 1 : Revenu total (paiements valides uniquement)
SELECT
    FORMAT(SUM(montant), 'N0') AS revenu_total_fcfa
FROM paiements
WHERE statut_paiement = 'Valide'
  AND montant > 0;


### INTERPRETATION

**Revenu total : 2 874 713 662 FCFA** sur 30 mois.
Converti en euros (1 EUR ≈ 656 FCFA) : environ **4,38 millions d'euros**.
Soit environ **146 000 EUR par mois** pour 5 hotels et 515 chambres — cohérent avec une chaine hoteliere 3-5 etoiles en Afrique de l'Ouest.

In [ ]:
-- KPI 2 : CSAT moyen par hotel
SELECT
    h.nom                                          AS hotel,
    h.etoiles,
    COUNT(r.reservation_id)                        AS nb_avis,
    ROUND(AVG(CAST(r.csat AS FLOAT)), 2)           AS csat_moyen,
    RANK() OVER (ORDER BY AVG(CAST(r.csat AS FLOAT)) DESC) AS rang
FROM reservations r
JOIN hotels h ON r.hotel_id = h.hotel_id
WHERE r.statut = 'Terminee'
  AND r.csat IS NOT NULL
GROUP BY h.hotel_id, h.nom, h.etoiles
ORDER BY csat_moyen DESC;


### INTERPRETATION

| Hotel | Etoiles | CSAT | Rang |
|---|---|---|---|
| HotelChain Cocody | 3 | **4.05** | 1 |
| HotelChain Dakar | 4 | 4.02 | 2 |
| HotelChain Douala | 3 | 4.01 | 3 |
| HotelChain Plateau | 4 | 4.00 | 4 |
| HotelChain Accra | 5 | 3.98 | 5 |

**Resultat contre-intuitif :** HotelChain Accra (5 etoiles) a le CSAT le plus bas (3.98) malgre son positionnement premium. Hypothese : les attentes des clients 5 etoiles sont plus elevees, donc plus difficiles a satisfaire pleinement. HotelChain Cocody (3 etoiles) surperforme car elle depasse les attentes d'un hotel 3 etoiles. **Signal d'alerte** pour Accra a creuser dans le NB2.

In [ ]:
-- KPI 3 : Duree moyenne de sejour et revenu moyen par hotel
SELECT
    h.nom                                           AS hotel,
    COUNT(r.reservation_id)                         AS nb_sejours,
    ROUND(AVG(CAST(r.nb_nuits AS FLOAT)), 2)        AS duree_moyenne_nuits,
    ROUND(AVG(r.montant_total), 0)                  AS revenu_moyen_sejour
FROM reservations r
JOIN hotels h ON r.hotel_id = h.hotel_id
WHERE r.statut IN ('Terminee','En cours')
  AND r.nb_nuits > 0
  AND r.montant_total > 0
GROUP BY h.hotel_id, h.nom
ORDER BY duree_moyenne_nuits DESC;


### INTERPRETATION

Duree moyenne globale : **2.69 nuits** par sejour. C'est une duree courte caracteristique des sejours d'affaires (le motif dominant dans les metropoles d'Afrique de l'Ouest). Un hotel touristique ou balneare aurait une duree de 4-7 nuits. Cette information est strategique : augmenter la duree moyenne d'une seule nuit augmenterait le revenu total de ~37% sans une seule reservation supplementaire.

In [ ]:
-- KPI 4 : Performance des canaux de reservation
SELECT
    r.canal,
    COUNT(*)                                    AS nb_reservations,
    ROUND(SUM(p.montant), 0)                    AS revenu_total,
    ROUND(AVG(p.montant), 0)                    AS revenu_moyen,
    ROUND(COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER(), 1)              AS part_marche_pct
FROM reservations r
LEFT JOIN paiements p
    ON r.reservation_id = p.reservation_id
    AND p.statut_paiement = 'Valide'
    AND p.montant > 0
WHERE r.statut IN ('Terminee','En cours')
GROUP BY r.canal
ORDER BY revenu_total DESC;


### INTERPRETATION

| Canal | Reservations | Revenu | Part marche |
|---|---|---|---|
| **Direct** | 2 406 | **963 M FCFA** | **30.1%** |
| Booking.com | 2 338 | 881 M FCFA | 29.2% |
| Agence voyage | 1 186 | 454 M FCFA | 14.8% |
| Expedia | 1 250 | 454 M FCFA | 15.6% |
| Corporate | 820 | 291 M FCFA | 10.3% |

**Insight cle :** Le canal Direct est le plus rentable avec 963 M FCFA et le revenu moyen par reservation le plus eleve. C'est logique : HotelChain West ne paie pas de commission sur les reservations directes (Booking.com et Expedia prennent entre 10-15%). Recommandation : investir dans le site web et la fidelisation pour augmenter la part du direct.

In [ ]:
-- KPI 5 : Repartition et prix par type de chambre
SELECT
    c.type_chambre,
    COUNT(DISTINCT c.chambre_id)                 AS nb_chambres,
    ROUND(AVG(c.prix_nuit), 0)                   AS prix_moyen_nuit,
    ROUND(COUNT(DISTINCT c.chambre_id) * 100.0
        / SUM(COUNT(DISTINCT c.chambre_id)) OVER(), 1) AS part_parc_pct
FROM chambres c
GROUP BY c.type_chambre
ORDER BY prix_moyen_nuit;


### INTERPRETATION

| Type | Nb chambres | Prix moyen/nuit | Part parc |
|---|---|---|---|
| Standard | 199 | 90 975 FCFA | 38.6% |
| Superieure | 157 | 138 803 FCFA | 30.5% |
| Deluxe | 110 | 209 455 FCFA | 21.4% |
| Suite | 49 | 402 500 FCFA | 9.5% |

La pyramide des chambres est bien construite : les Standard (les moins cheres) sont les plus nombreuses, les Suites (les plus cheres) sont rares. Le prix moyen d'une Suite est **4.4x** celui d'une Standard — un client Suite qui sejourne 3 nuits rapporte autant que 4 clients Standard sur le meme sejour.

---
## Bilan du Notebook 1

### Ce que nous avons accompli

| Etape | Resultat |
|---|---|
| SQL Server installe et connecte | OK |
| Base HotelChainDB creee | OK |
| 6 tables creees et chargees | OK |
| Schema de relations documente | OK |
| Anomalies identifiees | **6 types d'anomalies** |

### KPIs globaux HotelChain West

| KPI | Valeur |
|---|---|
| Revenu total (30 mois) | **2.87 Mrd FCFA** (~4.4M EUR) |
| CSAT moyen global | **4.01 / 5** |
| Duree moyenne sejour | **2.69 nuits** |
| Canal dominant | **Direct (30.1%)** |
| Taux annulation | **3.8%** |
| Revenu extras | **278 M FCFA** |

### Anomalies a corriger dans le Notebook 2

| Anomalie | Table | Nb | Action |
|---|---|---|---|
| Emails dupliques | clients | 3 | Exclure via vue |
| Ages aberrants (< 16 ou > 80) | clients | 4 | Exclure via vue |
| Montants negatifs | reservations | 5 | Filtrer montant > 0 |
| nb_nuits = 0 | reservations | 3 | Filtrer nb_nuits > 0 |
| Montants negatifs | paiements | 10 | Filtrer montant > 0 |

### Pour le Notebook 2
Nous allons creer des **vues SQL** pour encapsuler le nettoyage, puis construire les analyses avancees avec window functions (RANK, LAG, NTILE) et le calcul du **RevPAR** (indicateur standard de l'industrie hoteliere).

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.